In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.scoutai_common import get_engine, engineer_features, get_model, FULL_FEATURES, PERFORMANCE_FEATURES


def get_top_drivers(model_pipeline, n=3):
    try:
        importances = model_pipeline.named_steps["regressor"].feature_importances_
        feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()
        feat_imp = pd.Series(importances, index=feature_names)
        top = feat_imp.sort_values(ascending=False).head(n)
        return [(idx.replace("remainder__", "").replace("cat__", "").replace("_", " ").title(), val)
                for idx, val in top.items()]
    except Exception as e:
        print(f"[WARNING] Could not extract feature importance: {e}")
        return []


def analyze_player(player_name, engine):
    query = "SELECT * FROM view_scout_master WHERE player_name ILIKE %(pattern)s"
    player = pd.read_sql(query, engine, params={"pattern": f"%{player_name}%"})
    if player.empty:
        print(f"Player '{player_name}' not found.")
        return
    player = player.iloc[[0]].copy()
    player = engineer_features(player)

    has_history = bool(player["has_transfer_history"].iloc[0])
    model_label = "full" if has_history else "performance_only"
    feature_list = FULL_FEATURES if has_history else PERFORMANCE_FEATURES
    model_pipeline = get_model(model_label)

    pred_log = model_pipeline.predict(player[feature_list])
    pred_val = np.expm1(pred_log[0])
    actual_val = player["current_market_value"].iloc[0]
    c_name = player["club_name"].iloc[0] if "club_name" in player.columns else "Unknown"

    print(f"\n{'='*42}\n SCOUT AI PROFILE: {player['player_name'].iloc[0]}\n{'='*42}")
    print(f" Club:                 {c_name}")
    print(f" Transfer History:     {'Yes' if has_history else 'No (never transferred)'}")
    print(f" Model used:           {model_label}")
    print(f" Actual Market Value:  E{actual_val:,.0f}")
    print(f" ScoutAI Prediction:   E{pred_val:,.0f}")
    print(f" Gap (Value Add):      E{pred_val - actual_val:,.0f}")

    drivers = get_top_drivers(model_pipeline)
    if drivers:
        print("\n--- WHY THIS VALUATION? (Top 3 Drivers) ---")
        for clean_name, val in drivers:
            print(f"  * {clean_name:<25} : {val:.1%}")
    print("="*42 + "\n")

In [ ]:
engine = get_engine()
target = input("Enter player name to scout: ")
analyze_player(target, engine)